# 2 - interpretador de risco

Assina o tópico `flood/cemaden/raw`, cruza os dados de chuva com a previsão do Open-Meteo e calcula um índice de risco por estação. O resultado classificado é publicado em `flood/risco/classificado`.

**tópicos:**
- entrada: `flood/cemaden/raw`
- saída: `flood/risco/classificado`


In [1]:
import os
import json
import time
import requests
import paho.mqtt.client as mqtt
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

mqtt_broker = os.getenv("MQTT_BROKER")
mqtt_port = int(os.getenv("MQTT_PORT"))
topico_entrada = os.getenv("MQTT_TOPIC_RAW")
topico_saida = os.getenv("MQTT_TOPIC_RISCO")

openmeteo_url = "https://api.open-meteo.com/v1/forecast"


In [ ]:
def salvar_json(payload, origem="interpretador_risco"):
    # salva payload em landing/<origem>/<timestamp>.json
    ts = (
        payload.get("ts_processamento")
        or payload.get("ts_coleta", time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()))
    )
    pasta = Path("landing") / origem
    pasta.mkdir(parents=True, exist_ok=True)
    arquivo = pasta / f"{ts.replace(':', '-')}.json"
    with open(arquivo, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print(f"[landing] salvo em {arquivo}")

## consulta open-meteo

Pega precipitação prevista para as próximas 6 horas e umidade do solo atual no ponto da estação. A Open-Meteo não exige autenticação.


In [3]:
def buscar_openmeteo(lat, lon):
    # retorna precipitação prevista (próx. 6h) e umidade do solo atual
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "precipitation,soil_moisture_0_to_1cm",
        "forecast_days": 1,
        "timezone": "America/Sao_Paulo",
    }
    r = requests.get(openmeteo_url, params=params, timeout=10)
    r.raise_for_status()
    data = r.json()

    horas = data["hourly"]["time"]
    precip = data["hourly"]["precipitation"]
    soil = data["hourly"]["soil_moisture_0_to_1cm"]

    # localiza a hora atual na série e soma as próximas 6
    agora = time.strftime("%Y-%m-%dT%H:00", time.localtime())
    try:
        idx = horas.index(agora)
    except ValueError:
        idx = 0

    precip_6h = sum(p for p in precip[idx:idx+6] if p is not None)
    solo_atual = soil[idx] if soil[idx] is not None else 0.0

    salvar_json(data, origem="interpretador_risco/raw-open_meteo")

    return precip_6h, solo_atual


## cálculo do índice de risco

Pontuação de 0 a 100 baseada em limiares experimentais, a serem calibrados com dados históricos:

| variável | fonte | peso máximo |
|---|---|---|
| chuva acumulada 24h | CEMADEN | 40 pts |
| chuva acumulada 72h | CEMADEN | 20 pts |
| previsão precipitação 6h | Open-Meteo | 25 pts |
| umidade do solo | Open-Meteo | 15 pts |

Score final:

| score | classificação |
|---|---|
| 0-19 | BAIXO |
| 20-44 | MEDIO |
| 45-69 | ALTO |
| 70+ | CRITICO |


In [4]:
def calcular_risco(estacao, precip_6h, solo):
    score = 0

    c24 = estacao.get("chuva_24h") or 0
    if c24 >= 80: score += 40
    elif c24 >= 50: score += 25
    elif c24 >= 20: score += 10

    c72 = estacao.get("chuva_72h") or 0
    if c72 >= 150: score += 20
    elif c72 >= 80: score += 10

    if precip_6h >= 30: score += 25
    elif precip_6h >= 15: score += 15
    elif precip_6h >= 5: score += 5

    if solo >= 0.8: score += 15
    elif solo >= 0.5: score += 7

    if score >= 70: classe = "CRITICO"
    elif score >= 45: classe = "ALTO"
    elif score >= 20: classe = "MEDIO"
    else: classe = "BAIXO"

    return score, classe


## processamento da mensagem

Recebe o payload bruto do coletor, consulta o Open-Meteo uma vez (todas as estações estão em Florianópolis, coordenadas próximas), calcula o risco por estação e publica o resultado.


In [5]:
def processar(cliente, payload_raw):
    estacoes = payload_raw.get("estacoes", [])

    # usa coordenadas da primeira estação válida pra consultar o open-meteo
    ref = next((e for e in estacoes if e.get("lat") and e.get("lon")), None)
    if ref is None:
        print("[aviso] nenhuma estação com coordenadas, pulando")
        return

    try:
        precip_6h, solo = buscar_openmeteo(ref["lat"], ref["lon"])
        print(f"[open-meteo] precip. prev. 6h: {precip_6h:.1f}mm | solo: {solo:.2f}")
    except Exception as e:
        print(f"[aviso] open-meteo falhou: {e}. usando valores zero.")
        precip_6h, solo = 0.0, 0.0

    resultados = []
    for est in estacoes:
        score, classe = calcular_risco(est, precip_6h, solo)
        resultados.append({
            "codestacao": est["codestacao"],
            "nome": est["nome"],
            "lat": est["lat"],
            "lon": est["lon"],
            "timestamp": est["timestamp"],
            "chuva_1h": est.get("chuva_1h"),
            "chuva_24h": est.get("chuva_24h"),
            "chuva_72h": est.get("chuva_72h"),
            "precip_prev_6h": round(precip_6h, 2),
            "umidade_solo": round(solo, 3),
            "score_risco": score,
            "classificacao": classe,
        })

    payload_saida = {
        "ts_processamento": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "total_estacoes": len(resultados),
        "resultados": resultados,
    }

    salvar_json(payload_raw, origem="interpretador_risco/raw")
    salvar_json(payload_saida, origem="interpretador_risco/treated")

    msg = json.dumps(payload_saida, ensure_ascii=False)
    cliente.publish(topico_saida, msg, qos=1)
    print(f"[mqtt] publicado em '{topico_saida}' com {len(resultados)} estações")

    for cls in ["CRITICO", "ALTO", "MEDIO", "BAIXO"]:
        n = sum(1 for r in resultados if r["classificacao"] == cls)
        if n:
            print(f"  {cls}: {n} estações")


## loop mqtt

Fica em escuta permanente no tópico de entrada. Cada mensagem que chegar dispara o processamento completo.


In [6]:
def on_connect(cliente, userdata, flags, rc):
    if rc == 0:
        print(f"[mqtt] conectado. assinando '{topico_entrada}'...")
        cliente.subscribe(topico_entrada, qos=1)
    else:
        print(f"[mqtt] falha na conexão: rc={rc}")

def on_message(cliente, userdata, msg):
    print(f"\n[mqtt] mensagem recebida em '{msg.topic}'")
    try:
        payload = json.loads(msg.payload.decode("utf-8"))
        processar(cliente, payload)
    except json.JSONDecodeError as e:
        print(f"[erro] json inválido: {e}")

In [ ]:
cliente = mqtt.Client(client_id="interpretador-risco", protocol=mqtt.MQTTv311)
cliente.on_connect = on_connect
cliente.on_message = on_message

cliente.connect(mqtt_broker, mqtt_port, keepalive=60)
print("[info] interpretador iniciado. aguardando mensagens...\n")
cliente.loop_forever()


C:\Users\edupc\AppData\Local\Temp\ipykernel_55104\2642185850.py:1: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  cliente = mqtt.Client(client_id="interpretador-risco", protocol=mqtt.MQTTv311)


[info] interpretador iniciado. aguardando mensagens...

[mqtt] conectado. assinando 'flood/cemaden/raw'...

[mqtt] mensagem recebida em 'flood/cemaden/raw'
[landing] salvo em landing\interpretador_risco\raw-open_meteo\2026-06-15T21-40-32Z.json
[open-meteo] precip. prev. 6h: 0.0mm | solo: 0.28
[landing] salvo em landing\interpretador_risco\raw\2026-06-15T21-40-32Z.json
[landing] salvo em landing\interpretador_risco\treated\2026-06-15T21-40-32Z.json
[mqtt] publicado em 'flood/risco/classificado' com 5 estações
  BAIXO: 5 estações

[mqtt] mensagem recebida em 'flood/cemaden/raw'
[landing] salvo em landing\interpretador_risco\raw-open_meteo\2026-06-15T21-41-42Z.json
[open-meteo] precip. prev. 6h: 0.0mm | solo: 0.28
[landing] salvo em landing\interpretador_risco\raw\2026-06-15T21-41-42Z.json
[landing] salvo em landing\interpretador_risco\treated\2026-06-15T21-41-42Z.json
[mqtt] publicado em 'flood/risco/classificado' com 5 estações
  BAIXO: 5 estações

[mqtt] mensagem recebida em 'flood/cem